# DỰ ÁN 1: 🌍PHÂN TÍCH VÀ DỰ BÁO NHIỆT ĐỘ TOÀN CẦU (GLOBAL CLIMATE CHANGE)

## 1. Mục tiêu & câu hỏi dẫn đường

Data Understanding là bước chuyển từ câu hỏi **"Khoa học môi trường cần gì?"** sang câu hỏi **"Dữ liệu nào có thể giúp giải quyết nhu cầu đó?"**. Vì vậy, notebook này không bắt đầu bằng việc làm sạch hay huấn luyện mô hình, mà bắt đầu bằng việc hiểu dữ liệu đang mô tả phần nào của lịch sử khí hậu toàn cầu.

Trong bộ Global Climate Change, dữ liệu có thể được nhìn như một **tiến trình ghi nhận nhiệt độ lịch sử được ghi lại bằng bảng**: các trạm quan trắc ghi nhận nhiệt độ theo từng tháng, tổng hợp theo từng thành phố, quốc gia, lục địa qua hàng thế kỷ.

Đến cuối notebook, mỗi thành viên cần trả lời được 5 ý lớn:
1. **Bộ dữ liệu gồm những bảng nào** — bảng nhiệt độ toàn cầu, bảng nhiệt độ theo thành phố, quốc gia.
2. **Mỗi bảng đại diện cho phần nào trong không gian địa lý** — bảng này đang kể phần nào của câu chuyện nóng lên toàn cầu.
3. **Mỗi dòng trong bảng có ý nghĩa gì** — một tháng ghi nhận nhiệt độ tại một địa điểm cụ thể.
4. **Các bảng nối với nhau bằng khóa nào** (nếu có) hoặc cách tổng hợp dữ liệu (Aggregation) theo không gian.
5. **Dữ liệu có vấn đề gì cần chú ý** — quy mô lớn, chứa giá trị thiếu (missing values) ở các thế kỷ trước, kiểu dữ liệu tọa độ chưa chuẩn.

---

## 2. Bối cảnh bài toán & nguồn dữ liệu

### 2.1. Bối cảnh nghiệp vụ & nguồn dữ liệu

**Biến đổi khí hậu** đang là một trong những thách thức sinh tồn lớn nhất của nhân loại. Sự gia tăng nhiệt độ bề mặt Trái Đất đe dọa trực tiếp đến hệ sinh thái, mực nước biển và an ninh lương thực.

Nhóm nghiên cứu và các nhà lập chính sách đứng trước một bài toán nan giải:
* **Thiếu góc nhìn dài hạn** → dẫn đến việc phủ nhận biến đổi khí hậu hoặc đưa ra các chính sách sai lầm, ngắn hạn.
* **Dữ liệu thô quá phân tán** → khó có thể thấy được bức tranh toàn cảnh về việc khu vực nào đang nóng lên nhanh nhất.

Lối ra là **lượng hóa xu hướng nóng lên toàn cầu dựa trên dữ liệu lịch sử**: phân tích chuỗi thời gian (time-series) nhiệt độ đất liền và đại dương với độ phân giải cao từ năm 1750 đến nay. Đây chính là lý do bộ dữ liệu này được công bố, và cũng giải thích vì sao nó có tới **5 bảng dữ liệu** chia theo nhiều cấp độ địa lý (Toàn cầu, Quốc gia, Bang, Thành phố).

**Nguồn dữ liệu:** Bộ dữ liệu do tổ chức *Berkeley Earth* tổng hợp từ 1.6 tỷ báo cáo nhiệt độ, được chia sẻ trên Kaggle dưới tên *Climate Change: Earth Surface Temperature Data*. Bộ dữ liệu gồm **5 file CSV (~500 MB)**.

### 2.2. Phát biểu bài toán dưới dạng học máy

| Thành phần | Nội dung |
| :--- | :--- |
| **Đầu vào (features)** | Dữ liệu thời gian (Năm, Tháng) và Dữ liệu không gian (Vĩ độ, Kinh độ, Thành phố). |
| **Đầu ra (label)** | `AverageTemperature` ∈ ℝ — Nhiệt độ trung bình dự kiến (°C). |
| **Loại bài toán** | **Hồi quy, có giám sát** *(Supervised Regression) / Dự báo chuỗi thời gian (Time-Series Forecasting)* |
| **Chỉ số đánh giá** | **RMSE / MAE / R-squared** — đo lường sai số giữa nhiệt độ dự báo và thực tế. |

**Vì sao là Hồi quy chứ không phải bài toán khác:**
* **Không phải phân loại (Classification):** Thứ cần dự đoán không phải là **nhãn rời rạc** (nóng/lạnh), mà là một **đại lượng liên tục** (số độ C). Mô hình sẽ xuất ra giá trị thực tế cần dự báo.
* **Không phải phân cụm (Clustering):** Dữ liệu **đã có sẵn nhãn đúng** (`AverageTemperature`) cho các dữ liệu trong quá khứ. Có nhãn để học nghĩa là bài toán **có giám sát**.

**Các chỉ số cần phân biệt rõ:**
* **MAE (Sai số tuyệt đối trung bình):** Trung bình các khoảng lệch (tính bằng độ C) giữa dự báo và thực tế. Trực quan và dễ giải thích cho người ngoại đạo.
* **RMSE (Sai số bình phương trung bình căn):** Phạt nặng các dự báo sai lệch lớn. Rất quan trọng vì dự báo sai lệch nhiệt độ quá lớn có thể dẫn đến hậu quả nghiêm trọng trong đánh giá rủi ro khí hậu.
* **R-squared (Hệ số xác định):** Đo lường xem mô hình giải thích được bao nhiêu phần trăm sự biến thiên của nhiệt độ (từ 0 đến 1).
## 3. Hệ thống tập tin Dataset (CSV Files Overview)
Bộ dữ liệu **Climate Change: Earth Surface Temperature Data** được đóng gói thành các file có phân cấp từ toàn cầu đến địa phương. Dưới đây là bảng thống kê chi tiết các file dữ liệu thô có trong thư mục `data/raw/` của dự án:

| STT | Tên File (CSV) | Cấp độ phân giải | Mô tả chi tiết nội dung | Dung lượng ước tính |
| :---: | :--- | :--- | :--- | :---: |
| **1** | `GlobalTemperatures.csv` | Toàn cầu (Global) | Chứa thông tin nhiệt độ đất liền, nhiệt độ đại dương và các khoảng bất định toàn cầu từ năm 1750. | ~27.0 MB |
| **2** | `GlobalLandTemperaturesByCountry.csv` | Quốc gia (Country) | Nhiệt độ trung bình đất liền phân theo từng quốc gia trên thế giới từ năm 1743. | ~26.5 MB |
| **3** | `GlobalLandTemperaturesByState.csv` | Bang / Tỉnh (State) | Nhiệt độ chi tiết phân theo các vùng bang/tỉnh (đặc biệt hữu ích cho các quốc gia lớn như Mỹ, Úc). | ~54.0 MB |
| **4** | `GlobalLandTemperaturesByMajorCity.csv` | Thành phố lớn (Major City) | Tập trung vào các đô thị lớn tiêu biểu có lịch sử quan trắc khí hậu lâu đời. | ~3.0 MB |
| **5** | `GlobalLandTemperaturesByCity.csv` | Thành phố (City / Fact Table) | **Bảng chính (Fact Table):** Chứa dữ liệu chi tiết của hàng nghìn thành phố trên toàn cầu (hơn 8.5 triệu dòng). | ~335.0 MB |

> 💡 **Ghi chú kỹ thuật:** Tổng dung lượng toàn bộ thư mục dữ liệu thô khoảng **~445 MB**. Do bảng số 5 (`GlobalLandTemperaturesByCity.csv`) có số lượng bản ghi rất lớn, nhóm sẽ sử dụng kỹ thuật quản lý bộ nhớ (`gc.collect()`) hoặc đẩy sang PostgreSQL ở Notebook 02 để xử lý nhằm tránh tình trạng tràn RAM (Out-of-Memory).

## 3. Chuẩn bị môi trường

Thiết lập thư viện, cấu hình hiển thị và đường dẫn tới dữ liệu thô.

**Vai trò của từng thư viện trong notebook này:**

| Thư viện | Vai trò |
| :--- | :--- |
| `pandas` | Đọc CSV và thao tác dữ liệu dạng bảng (DataFrame) — công cụ chính của toàn bộ notebook. |
| `numpy` | Nền tảng tính toán số học mà pandas dựa lên; ở notebook này chủ yếu dùng gián tiếp. |
| `matplotlib` / `seaborn` | Vẽ biểu đồ. `seaborn` là lớp bọc trên `matplotlib`, cho cú pháp ngắn gọn hơn với biểu đồ thống kê. |
| `gc` (garbage collector) | Thu hồi bộ nhớ ngay sau khi giải phóng một bảng lớn — cần thiết vì dữ liệu thô khá lớn. |
| `pathlib.Path` | Biểu diễn đường dẫn độc lập hệ điều hành, giúp notebook chạy được trên cả Windows và Linux. |

📌 **Về cấu hình hiển thị:** đặt `display.max_columns = 120` để pandas không rút gọn cột khi in bảng; `display.float_format` để không hiển thị số dạng khoa học (VD: 1.5e6).

---

## 4. Bản đồ dữ liệu: các bảng và quy mô

Quét lần lượt từng bảng để lấy: số dòng, số cột, dung lượng khi nạp vào RAM, tổng số ô thiếu và tỷ lệ thiếu.

**Vì sao nạp rồi giải phóng ngay từng bảng:** Tổng dữ liệu thô khá lớn, và khi nạp vào RAM còn phình to hơn (pandas phải cấp phát thêm cho chỉ mục và chuỗi ký tự). Nếu giữ cả 5 bảng cùng lúc trong bộ nhớ, máy có RAM vừa phải có thể bị tràn bộ nhớ (Out-Of-Memory). Vì mục này chỉ cần *thống kê tổng quan* chứ không cần giữ dữ liệu, hàm `profile_table` nạp một bảng → tính xong các chỉ số → `del` và `gc.collect()` để trả bộ nhớ về hệ điều hành trước khi sang bảng kế tiếp. Nhờ vậy đỉnh RAM chỉ bằng bảng lớn nhất, không phải tổng 5 bảng.

**Các chỉ số được đo và ý nghĩa:**

| Chỉ số | Ý nghĩa |
| :--- | :--- |
| Số dòng / Số cột | Kích thước bảng. Chênh lệch số dòng giữa các bảng cho biết cấp độ chi tiết (một dòng = một tháng của một thành phố, hay của toàn cầu). |
| Cột số / Cột chữ | Cột chữ là biến phân loại hoặc tọa độ, sẽ phải mã hóa thành số hoặc tiền xử lý trước khi đưa vào mô hình (notebook 02 & 05). |
| RAM (MB) | Dung lượng thật khi bảng nằm trong bộ nhớ — căn cứ để ước lượng máy cần bao nhiêu RAM. |
| Ô thiếu (%) | Tỷ lệ ô trống trên tổng số ô. Đây là căn cứ đầu tiên để lên kế hoạch làm sạch ở notebook 03. |

In [1]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Cấu hình hiển thị
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

# Đường dẫn dữ liệu
DATA_RAW = next((p for p in [Path("data/raw"), Path("../data/raw")] if p.exists()), Path("../data/raw"))

# Danh sách các bảng
TABLES = {
    "GlobalTemperatures.csv": "Nhiệt độ toàn cầu",
    "GlobalLandTemperaturesByCountry.csv": "Nhiệt độ theo Quốc gia",
    "GlobalLandTemperaturesByState.csv": "Nhiệt độ theo Bang",
    "GlobalLandTemperaturesByMajorCity.csv": "Nhiệt độ theo Thành phố lớn",
    "GlobalLandTemperaturesByCity.csv": "Nhiệt độ theo Thành phố (Bảng chính)"
}

def profile_table(path):
    """Nạp 1 bảng, trích xuất thống kê tổng quan rồi giải phóng bộ nhớ."""
    df = pd.read_csv(path)
    n_rows, n_cols = df.shape
    
    # Đo dung lượng RAM thực tế
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2 
    
    # Tính tỷ lệ thiếu
    total_cells = n_rows * n_cols
    missing_cells = int(df.isna().sum().sum())
    missing_pct = missing_cells / total_cells * 100 if total_cells else 0 
    
    # Đếm loại biến
    n_numeric = df.select_dtypes(include="number").shape[1]
    n_object = df.select_dtypes(include="object").shape[1]
    
    # Tránh tràn RAM
    del df
    gc.collect()
    
    return {
        "Số dòng": n_rows,
        "Số cột": n_cols,
        "Cột số": n_numeric,
        "Cột chữ": n_object,
        "RAM (MB)": round(mem_mb, 1),
        "Ô thiếu (%)": round(missing_pct, 2),
    }

print("Đang quét thư mục:", DATA_RAW.resolve())
rows = {}
for fname in TABLES:
    p = DATA_RAW / fname
    if p.exists():
        rows[fname] = profile_table(p)
        print(f"✅ Đã quét: {fname}")
    else:
        print(f"❌ KHÔNG TÌM THẤY: {fname}")

# Tạo DataFrame hiển thị
overview = pd.DataFrame(rows).T
overview.index.name = "Bảng"
# Hiển thị trực tiếp vì môi trường hiện tại không nhất thiết có jinja2 cho Styler
display(overview)

Đang quét thư mục: C:\Global Climate Change\data\raw
✅ Đã quét: GlobalTemperatures.csv
✅ Đã quét: GlobalLandTemperaturesByCountry.csv
✅ Đã quét: GlobalLandTemperaturesByState.csv
✅ Đã quét: GlobalLandTemperaturesByMajorCity.csv
✅ Đã quét: GlobalLandTemperaturesByCity.csv


,Số dòng,Số cột,Cột số,Cột chữ,RAM (MB),Ô thiếu (%)
Bảng,,,,,,
GlobalTemperatures.csv,"3,192.0000",9.0000,8.0000,1.0000,0.4000,25.1500
GlobalLandTemperaturesByCountry.csv,"577,462.0000",4.0000,2.0000,2.0000,73.7000,2.8000
GlobalLandTemperaturesByState.csv,"645,675.0000",5.0000,2.0000,3.0000,116.7000,1.5900
GlobalLandTemperaturesByMajorCity.csv,"239,177.0000",7.0000,2.0000,5.0000,67.9000,1.3100
GlobalLandTemperaturesByCity.csv,"8,599,212.0000",7.0000,2.0000,5.0000,"2,455.2000",1.2100


**Nhận xét:**
* Bộ dữ liệu gồm 5 bảng với quy mô rất khác nhau: `GlobalLandTemperaturesByCity` là bảng trung tâm (mỗi dòng = 1 tháng ghi nhận tại 1 thành phố) chứa số lượng bản ghi lớn nhất (hơn 8.5 triệu dòng).
* Tỷ lệ ô thiếu dao động quanh 1-3%. Tuy tỷ lệ không quá lớn, nhưng do tính chất chuỗi thời gian, ta không thể xóa bỏ (drop) một cách tùy tiện mà cần lên kế hoạch xử lý (Interpolation/Forward Fill) ở notebook 03 (Data Cleaning).

---

## 5. Bảng trung tâm — `GlobalLandTemperaturesByCity`

Đây là bảng dùng để huấn luyện mô hình. Ta nạp riêng và phân tích sâu: kiểu dữ liệu, biến mục tiêu, giá trị thiếu và thống kê mô tả.

### 5.1. Kích thước & kiểu dữ liệu

*Kiểu dữ liệu (dtype)* quyết định cột được xử lý ra sao ở các bước sau. Ba nhóm gặp trong bảng này:
* `int64` — số nguyên (số con, số ngày).
* `float64` — số thực; **mọi cột có ô thiếu đều buộc phải mang kiểu này**, vì `NaN` chỉ tồn tại ở dạng số thực. Đây là lý do nhiều cột đếm vẫn hiện `float64`.
* `str` — chuỗi ký tự, tức **biến phân loại** hoặc **tọa độ** (Tên thành phố, Quốc gia, Vĩ độ, Kinh độ). Mô hình học máy chỉ nhận số, nên nhóm tọa độ phải được **tiền xử lý (loại bỏ N/S/E/W)** ở notebook 02, và nhóm phân loại phải được **mã hóa (encoding)** ở notebook 05.

📌 **Lưu ý phiên bản:** từ pandas 3.0, cột chữ được báo là `str` thay vì `object` như các phiên bản trước, nên bảng đếm dtype bên dưới hiện `str`. Truy vấn `select_dtypes(include="object")` vẫn bắt đúng các cột này để giữ tương thích ngược.

In [2]:
main_file = DATA_RAW / "GlobalLandTemperaturesByCity.csv"
print(f"Đang nạp bảng chính: {main_file.name}...\n")
df_main = pd.read_csv(main_file)

print("Kích thước:", df_main.shape)
print("\nPhân loại kiểu dữ liệu:")
print(df_main.dtypes.value_counts())

print("\n--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df_main.head())

print("\n--- THỐNG KÊ SỐ LƯỢNG GIÁ TRỊ KHÁC NHAU (UNIQUE) ---")
unique_df = pd.DataFrame({
    "Unique Count": df_main.nunique(), 
    "Tỷ lệ Unique (%)": (df_main.nunique() / len(df_main) * 100).round(4)
})
display(unique_df)


Đang nạp bảng chính: GlobalLandTemperaturesByCity.csv...

Kích thước: (8599212, 7)

Phân loại kiểu dữ liệu:
str        5
float64    2
Name: count, dtype: int64

--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---


,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1743-11-01,6.0680,1.7370,Århus,Denmark,57.05N,10.33E
1,1743-12-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
2,1744-01-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
3,1744-02-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
4,1744-03-01,NaN,NaN,Århus,Denmark,57.05N,10.33E



--- THỐNG KÊ SỐ LƯỢNG GIÁ TRỊ KHÁC NHAU (UNIQUE) ---


,Unique Count,Tỷ lệ Unique (%)
dt,3239,0.0377
AverageTemperature,103481,1.2034
AverageTemperatureUncertainty,10902,0.1268
City,3448,0.0401
Country,159,0.0018
Latitude,73,0.0008
Longitude,1227,0.0143


### 5.2. Biến mục tiêu `AverageTemperature`

`AverageTemperature` là cột mà mô hình phải học để dự đoán. Nghĩa là chúng ta muốn biết nhiệt độ trung bình tại một thành phố vào một thời điểm trong tương lai sẽ là bao nhiêu.

**Nhận xét về biến mục tiêu:**
* Đây là một phân bố liên tục. Việc phân tích phân bố (Distribution) giúp ta xác định xem nhiệt độ có bị lệch (Skewed) hay chứa giá trị ngoại lai (Outliers) không.
* Cột mục tiêu `AverageTemperature` cũng chứa giá trị khuyết thiếu. Tại những dòng mà biến mục tiêu bị thiếu, ta không thể dùng nó để huấn luyện mô hình.

### 5.3. Giá trị thiếu (missing values)

Ô thiếu (`NaN` — *Not a Number*) là ô không có dữ liệu. Đa số thuật toán học máy không chấp nhận `NaN`. Ô thiếu đều phải được xử lý trước khi huấn luyện: hoặc **điền khuyết (imputation)** bằng một giá trị suy ra từ thống kê, hoặc **loại bỏ dòng/cột** nếu thiếu quá nhiều đến mức không còn thông tin để suy.

Về đoạn code bên dưới: `app.isna()` trả về một bảng True/False cùng kích thước (True = ô trống); `.sum()` cộng theo từng cột, cho ra số ô thiếu của mỗi cột. Sau đó lọc `miss > 0` để chỉ giữ các cột thật sự có thiếu, và chia cho `len(app)` để đổi số đếm thành tỷ lệ phần trăm — tỷ lệ mới là thứ dễ so sánh đánh giá.

⚠️ **Lưu ý phương pháp:** Trong dự án Khí hậu, vì dữ liệu có tính chất chuỗi thời gian, việc **xóa dòng (dropna)** có thể làm đứt gãy chuỗi. Cần xem xét áp dụng kỹ thuật Forward Fill hoặc Nội suy (Interpolation) ở Notebook 03.

In [3]:
print("--- KIỂM TRA DỮ LIỆU BỊ KHUYẾT (MISSING VALUES) ---")

miss = df_main.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_pct = (miss / len(df_main) * 100).round(2)

miss_table = pd.DataFrame({
    "Số lượng ô thiếu": miss, 
    "Tỷ lệ thiếu (%)": miss_pct
})

print(f"\nSổ cột bị thiếu: {len(miss)} / {df_main.shape[1]}")
display(miss_table)

print("\n--- THỐNG KÊ MÔ TẢ (DESCRIBE) ---")
display(df_main.describe().T)

--- KIỂM TRA DỮ LIỆU BỊ KHUYẾT (MISSING VALUES) ---

Sổ cột bị thiếu: 2 / 7


,Số lượng ô thiếu,Tỷ lệ thiếu (%)
AverageTemperature,364130,4.2300
AverageTemperatureUncertainty,364130,4.2300



--- THỐNG KÊ MÔ TẢ (DESCRIBE) ---


,count,mean,std,min,25%,50%,75%,max
AverageTemperature,"8,235,082.0000",16.7274,10.3534,-42.7040,10.2990,18.8310,25.2100,39.6510
AverageTemperatureUncertainty,"8,235,082.0000",1.0286,1.1297,0.0340,0.3370,0.5910,1.3490,15.3960
